# BUGS COMUNES — Encuentra el error

Si el profe te da un script y dice *"hay cosas mal, arréglalas"*, busca estos patrones **en este orden**.

Cada sección: **❌ MAL** → **✅ BIEN** → **Por qué**.

## Índice de bugs
1. Leakage del scaler (fit sobre todos los datos)
2. Split aleatorio en datos temporales
3. Falta `stratify` en clasificación
4. Accuracy en datos desbalanceados
5. Imputar la media antes del split
6. `dropna()` que tira el dataset
7. Edades / valores imposibles no tratados
8. Columna `id` usada como feature
9. GridSearch evaluado en test
10. Falta `handle_unknown='ignore'` en OneHotEncoder
11. PyTorch: olvidar `model.eval()` y `torch.no_grad()`
12. PyTorch: target en regresión sin escalar
13. PyTorch: usar MSE en clasificación
14. Olvidar `class_weight='balanced'` con desbalance
15. Predecir con `predict_proba` en regresión / al revés
16. Falta `zero_division=0` en classification_report
17. Variables categóricas sin codificar

## 1. Leakage del scaler (MUY COMÚN)

In [ ]:
# ❌ MAL — fit sobre todo X (train + test mezclados)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)        # ← el test "sabe" la media de todo
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

# ✅ BIEN — split primero, fit solo en train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # ← fit SOLO con train
X_test_s  = scaler.transform(X_test)       # ← transform sin re-fit

# ✅ MEJOR — usar Pipeline que ya hace esto bien
# Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor())])

# Por qué: si el scaler ve el test, las stats (media/std) están contaminadas.
# Las métricas de test salen optimistas y no reflejan la realidad en producción.

## 2. Split aleatorio en datos temporales

In [ ]:
# ❌ MAL — datos con fecha, split aleatorio
df = pd.read_csv("kc_house_data.csv")  # tiene columna 'date'
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
# El train contiene ventas de dic-2014 y el test de ene-2014 → leakage temporal

# ✅ BIEN — ordenar por fecha y split por posición
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]   # 80% más antiguo
test_df  = df.iloc[split_idx:]   # 20% más reciente

# Por qué: predecir el pasado conociendo el futuro es trampa.
# Para CV usa TimeSeriesSplit, no KFold.

## 3. Falta `stratify` en clasificación

In [ ]:
# ❌ MAL — clases desbalanceadas, split sin stratify
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# El test puede quedar con 0 muestras de la clase rara

# ✅ BIEN
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # ← mantiene proporciones
)

# Por qué: en clasificación, train y test deben tener la misma distribución de clases.
# Sin stratify, en datasets pequeños puede haber clases ausentes en test.

## 4. Accuracy en datos desbalanceados

In [ ]:
# ❌ MAL — 95% de la clase 'sano', el modelo predice siempre 'sano' y la accuracy es 95%
from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")  # 95% pero el modelo es inútil

# ✅ BIEN — usar F1 macro / recall por clase / classification_report
from sklearn.metrics import classification_report, f1_score
print(classification_report(y_test, y_pred, zero_division=0))
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro'):.4f}")

# Por qué: accuracy promedia sobre TODAS las muestras. Si la clase mayoritaria
# domina, la accuracy oculta que el modelo ignora a la minoritaria.

## 5. Imputar la media antes del split

In [ ]:
# ❌ MAL — calcular la mediana con todos los datos
df["age"] = df["age"].fillna(df["age"].median())  # ← mediana de train+test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ✅ BIEN — usar SimpleImputer DENTRO del Pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # fit solo en train
    ("scaler",  StandardScaler()),
    ("model",   RandomForestRegressor()),
])
pipe.fit(X_train, y_train)

# Por qué: la mediana es una estadística. Si la calculas con el test, el test
# contaminó el preprocesado del train. Mismo problema que el scaler.

## 6. `dropna()` que tira el dataset

In [ ]:
# ❌ MAL — borrar todas las filas con cualquier NaN
df = df.dropna()  # puedes pasar de 10000 filas a 200

# ✅ BIEN — imputar features, solo borrar NaN en el TARGET
df = df.dropna(subset=[TARGET])   # NaN en y sí se quitan (no hay nada que aprender)
# El resto de NaN se imputan en el pipeline

# Por qué: una fila con NaN en 1 columna sigue teniendo info útil en las otras.

## 7. Edades o valores imposibles no tratados

In [ ]:
# ❌ MAL — Thyroid tiene edades de 455 años porque alguien metió un código en age
df["age"].describe()  # max = 455 → outlier que rompe el modelo

# ✅ BIEN — tratar valores imposibles como NaN
import numpy as np
df.loc[df["age"] > 120, "age"] = np.nan   # luego el imputer los rellena

# Por qué: un outlier extremo desplaza la media/std y rompe el StandardScaler.

## 8. Columna `id` usada como feature

In [ ]:
# ❌ MAL — el id es solo un identificador, no aporta info
X = df.drop(columns=["price"])   # ← se queda 'id' dentro

# ✅ BIEN — quitar columnas que solo memorizan
X = df.drop(columns=["price", "id"])
# También: patient_id, user_id, timestamp si no es feature temporal, etc.

# Por qué: el modelo puede memorizar id→target en train y fallar en test.
# Da R²=1.0 en train, 0.3 en test. Síntoma típico de overfitting por id.

## 9. GridSearch evaluado en test

In [ ]:
# ❌ MAL — probar 20 modelos en test y reportar el mejor
for params in todas_las_combinaciones:
    model.set_params(**params)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)   # ← test usado para elegir

# ✅ BIEN — GridSearch usa CV internamente, test solo al final
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(model, param_grid, cv=5, scoring="neg_root_mean_squared_error")
grid.fit(X_train, y_train)                # solo train
print(grid.best_params_)
final_score = grid.score(X_test, y_test)  # ← UNA SOLA VEZ al final

# Por qué: si eliges hiperparámetros mirando test, ya no es test, es validation.
# Las métricas reportadas dejan de ser una estimación honesta.

## 10. Falta `handle_unknown='ignore'` en OneHotEncoder

In [ ]:
# ❌ MAL — si en test aparece una categoría no vista en train, peta
encoder = OneHotEncoder()  # por defecto handle_unknown='error'
# ValueError: Found unknown categories ['Z'] in column 0 during transform

# ✅ BIEN
encoder = OneHotEncoder(handle_unknown="ignore")  # ← codifica como todo ceros

# Por qué: en producción siempre llegan categorías nuevas. Ignorar es más robusto.

## 11. PyTorch: olvidar `model.eval()` y `torch.no_grad()`

In [ ]:
# ❌ MAL
preds = []
for X_b, _ in test_loader:
    preds.append(model(X_b))      # Dropout sigue activo → predicciones ruidosas
                                  # se guardan gradientes → gasta memoria

# ✅ BIEN
model.eval()                      # desactiva Dropout, BatchNorm usa stats globales
with torch.no_grad():             # no guarda gradientes
    for X_b, _ in test_loader:
        preds.append(model(X_b))

# Por qué: Dropout aleatorio → métricas inconsistentes en validation.
# Sin no_grad → memoria llena, todo más lento.

## 12. PyTorch regresión: target sin escalar

In [ ]:
# ❌ MAL — y son precios entre 100000 y 5000000
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
# El loss MSE será del orden de 10^10 → gradientes explotan → loss=NaN

# ✅ BIEN — escalar con log1p + StandardScaler
import numpy as np
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(np.log1p(y_train.values).reshape(-1, 1))
# Para predecir en unidades reales:
# y_real = np.expm1(y_scaler.inverse_transform(preds))

# Por qué: las NN trabajan mejor con targets ~N(0, 1).

## 13. PyTorch: usar MSE en clasificación

In [ ]:
# ❌ MAL
criterion = nn.MSELoss()           # para regresión
# ...
loss = criterion(logits, y_b.float())  # con etiquetas tipo 0,1,2 no tiene sentido

# ✅ BIEN — CrossEntropyLoss para clasificación
criterion = nn.CrossEntropyLoss()   # incluye Softmax internamente
# y_b debe ser long (enteros), no float
loss = criterion(logits, y_b.long())
# logits shape: (batch, n_clases), sin softmax aplicado

# Por qué: MSE trata las clases como números (clase 2 está "el doble de lejos" que clase 1).
# CrossEntropy las trata como categorías independientes.

## 14. Olvidar `class_weight='balanced'`

In [ ]:
# ❌ MAL — Thyroid 90% 'sano', 10% 'enfermo'. El modelo predice siempre 'sano'.
model = RandomForestClassifier()
model.fit(X_train, y_train)
# accuracy=0.90, recall en 'enfermo' = 0.0

# ✅ BIEN
model = RandomForestClassifier(class_weight="balanced")
# o LogisticRegression(class_weight="balanced"), SVC(class_weight="balanced")...

# En PyTorch:
import numpy as np
counts = np.bincount(y_train)
w = torch.tensor(len(y_train) / (len(counts) * counts), dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=w.to(device))

## 15. Confundir `predict` y `predict_proba`

In [ ]:
# ❌ MAL — querías la probabilidad pero pediste la clase
umbral = 0.7
y_pred = model.predict(X_test)       # devuelve 0/1, no probabilidades
y_pred_custom = (y_pred > umbral)    # casi siempre todo False, no tiene sentido

# ✅ BIEN
y_proba = model.predict_proba(X_test)[:, 1]   # proba de la clase positiva
y_pred = (y_proba > umbral).astype(int)

# Y al revés: para roc_auc necesitas probabilidades, no clases
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, y_proba)   # ← probabilidades
# NO: roc_auc_score(y_test, y_pred)

## 16. `zero_division` warning

In [ ]:
# ❌ MAL — warnings que ensucian la salida
print(classification_report(y_test, y_pred))
# UndefinedMetricWarning: Precision is ill-defined and being set to 0.0

# ✅ BIEN — pasar zero_division=0 explícitamente
print(classification_report(y_test, y_pred, zero_division=0))

# Por qué: si el modelo nunca predice una clase, precision = 0/0.
# Sklearn lo marca como 0 pero avisa. Mejor decirle que sí, es 0.

## 17. Variables categóricas sin codificar

In [ ]:
# ❌ MAL
model.fit(X_train, y_train)
# ValueError: could not convert string to float: 'NEAR BAY'

# ✅ BIEN — ColumnTransformer separa por tipo
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

# Alternativa rápida (no recomendada en pipeline): pd.get_dummies(X_train)

## Checklist de revisión rápida

Si el profe te da un script, revisa **EN ESTE ORDEN**:

1. ¿`fit_transform` aparece sobre algo que NO sea solo train? → bug #1, #5
2. ¿Hay columna de fecha y se usa `train_test_split` aleatorio? → bug #2
3. ¿Es clasificación? ¿Falta `stratify=y`? → bug #3
4. ¿Se reporta solo `accuracy` con clases desbalanceadas? → bug #4
5. ¿Hay `dropna()` sin `subset=`? → bug #6
6. ¿Hay columna `id`, `patient_id`, etc. en `X`? → bug #8
7. ¿Se evalúa en test más de una vez? → bug #9
8. ¿OneHotEncoder sin `handle_unknown="ignore"`? → bug #10
9. ¿Si hay PyTorch: `model.eval()` antes de predecir? → bug #11
10. ¿Si hay desbalance: `class_weight="balanced"`? → bug #14